<!-- torchleet:colab -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Exorust/TorchLeet/blob/main/llm/SmolLM/smollm-q12-Question.ipynb)

Check your work: `!pip install torchleet` then `from torchleet import check; check("smollm", ...)`


In [2]:
import math
import torch
from torch import nn
import torch.nn.functional as F

In [112]:
# ********************************************************* Helper functions ************************************************
def rotate_half(x):
    x1, x2 = x[..., :x.shape[-1]//2], x[..., x.shape[-1]//2:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin, position_ids=None, unsqueeze_dim=1):
    # Expand cos and sin tensors for broadcasting
    return q * cos.unsqueeze(0).unsqueeze(0) + rotate_half(q) * sin.unsqueeze(0).unsqueeze(0), k * cos.unsqueeze(0).unsqueeze(0) + rotate_half(k) * sin.unsqueeze(0).unsqueeze(0)

def repeat_kv(hidden_states, n_rep):
    hidden_states = hidden_states.repeat_interleave(n_rep, dim = 1)
    return hidden_states

In [113]:
# Computes rotary positional embeddings for queries and keys
class RotaryEmbedder(nn.Module):
    def __init__(self, dim, base):
        super().__init__()
        self.dim = dim
        self.base = base
        inv_freq = 1 / (base**(torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        self.seq_len_cached = None
        self.sin = None
        self.cos = None

    @torch.no_grad()
    def forward(self, x):
        seq_len = x.shape[2]
        if seq_len != self.seq_len_cached:
            self.seq_len_cached = seq_len
            m = torch.arange(x.shape[2], device = x.device).type_as(self.inv_freq)
            freqs = torch.einsum('i,j->ij', m, self.inv_freq)
            emb = torch.cat((freqs, freqs), dim=-1).to(x.device)
            self.sin = torch.sin(emb)
            self.cos = torch.cos(emb)
        return self.cos, self.sin


# Implements attention with rotary positional embeddings
class RopeAttention(nn.Module):
    def __init__(self, config):
      super().__init__()

      self.hidden_size = config.hidden_size
      self.num_heads = config.num_heads
      self.head_dim = config.hidden_size // self.num_heads
      self.kv_heads = config.kv_heads
      self.rope_theta = 10000.0

      self.W_q = nn.Linear(config.hidden_size, self.num_heads * self.head_dim, bias = False)
      self.W_k = nn.Linear(config.hidden_size, self.kv_heads * self.head_dim, bias = False)
      self.W_v = nn.Linear(config.hidden_size, self.kv_heads * self.head_dim, bias = False)
      self.W_o = nn.Linear(config.hidden_size, config.hidden_size, bias = False)

      self.rotary_emb = RotaryEmbedder(base=self.rope_theta, dim=self.head_dim)

    def forward(self, hidden_states: torch.Tensor, attention_mask=None):
        # Input dimensions: (batch_size, seq_len, hidden_size)
        # here hidden_states is input x
        batch_size, seq_len, _ = hidden_states.shape

        q = self.W_q(hidden_states).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1,2)
        k = self.W_k(hidden_states).view(batch_size, seq_len, self.kv_heads, self.head_dim).transpose(1,2)
        v = self.W_v(hidden_states).view(batch_size, seq_len, self.kv_heads, self.head_dim).transpose(1,2)

        cos, sin = self.rotary_emb(q)
        # print('rotatory wokring fine')

        k = repeat_kv(k, self.num_heads//self.kv_heads)
        q_rot, k_rot = apply_rotary_pos_emb(q, k, cos, sin)
        # print('apply_rotary_pos_emb working fine')
        v = repeat_kv(v, self.num_heads//self.kv_heads)
        k_rot = repeat_kv(k_rot, self.num_heads//self.kv_heads)
        # print('reapeat_kv working fine')

        scores = q_rot @ k.transpose(-2,-1) / torch.sqrt(torch.tensor(self.head_dim, dtype = torch.float32))

        if attention_mask is not None:
            scores = scores + attention_mask

        logits = F.softmax(scores, dim = -1)

        attn_output = (logits @ v).transpose(1,2).contiguous().view(batch_size, seq_len, self.num_heads * self.head_dim)

        out = self.W_o(attn_output)
        # print('rope working fine')
        return out

In [114]:
class MLP(nn.Module):
  def __init__(self, inp_dim, hidden_dim):
    super().__init__()
    self.input_layer = nn.Linear(inp_dim, hidden_dim)
    self.hidden_layer = nn.Linear(hidden_dim, hidden_dim)
    self.output_layer = nn.Linear(hidden_dim, inp_dim)
    self.activation = nn.ReLU()

  def forward(self, x):
    return self.activation(self.output_layer(self.hidden_layer(self.input_layer(x))))


class RMSNorm(nn.Module):
  def __init__(self, hidden_size, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))  # Learnable scaling factor
        self.variance_epsilon = eps

  def forward(self, hidden_states):
        # Calculate variance along the last dimension (hidden size)
        variance = hidden_states.pow(2).mean(-1, keepdim=True)

        # Normalize and scale
        hidden_states = hidden_states * torch.rsqrt(variance + self.variance_epsilon)
        return self.weight * hidden_states

In [115]:
class Decoder(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.rope_attention = RopeAttention(config)
    self.mlp = MLP(config.hidden_size, config.intermediate_size)
    self.pre_attn_norm = RMSNorm(config.hidden_size)
    self.pre_mlp_norm = RMSNorm(config.hidden_size)

  def forward(self, hidden_states, attention_mask = None):
    # print('entering decoder')
    if attention_mask is not None:
      attention_mask = torch.triu(torch.full((attention_mask.shape[-1], attention_mask.shape[-1]),fill_value=float('-inf')), diagonal=1)
    # print('attention mask created')
    hidden_states = hidden_states + self.rope_attention(self.pre_attn_norm(hidden_states), attention_mask)
    # print('rope attention working fine')
    hidden_states = hidden_states + self.mlp(self.pre_mlp_norm(hidden_states))
    # print('mlp working fine')
    return hidden_states

In [116]:
class smolModel(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.embed_tokens = nn.Embedding(
            num_embeddings=config.vocab_size,
            embedding_dim=config.hidden_size
        )
    self.layers = nn.ModuleList([
            Decoder(config) for _ in range(config.num_hidden_layers)
        ])
    self.norm = RMSNorm(config.hidden_size, eps=1e-05)

  def forward(self, input_ids, attention_mask = None):
    # print('entering model')
    hidden_states = self.embed_tokens(input_ids)
    # print('embedding working fine')
    for layer in self.layers:
      hidden_states = layer(hidden_states, attention_mask)
    # print('decoder layers working fine')
    return self.norm(hidden_states)

In [117]:
class smolLM(nn.Module):
  def __init__(self, config):
    super().__init__()
    self.model = smolModel(config)
    self.output_layer = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
    self.output_layer.weight = self.model.embed_tokens.weight

  def forward(self, input_ids, attention_mask = None):
    hidden_states = self.model(input_ids, attention_mask)
    logits = self.output_layer(hidden_states).float()
    return {'logits': logits}

TEST

In [85]:
from transformers import AutoTokenizer, AutoModelForCausalLM


# Libraries
import torch
import torch.nn.functional as F
from torch import nn
import math

########################## HELPER FUNCTIONS ######################

def __generate(model, inputs, num_tokens, tokenizer, max_length=50):
    collect = []
    for _ in range(num_tokens):
        output = model(**inputs)
        output_id = torch.argmax(output['logits'][0, -1]).item()
        collect.append(output_id)
        if output_id == tokenizer.eos_token_id or len(collect) >= max_length:
            break
        # Update input_ids and attention_mask
        new_token = torch.tensor([output_id], device=inputs['input_ids'].device)
        inputs['input_ids'] = torch.cat([inputs['input_ids'][0], new_token]).unsqueeze(0)
        inputs['attention_mask'] = F.pad(inputs['attention_mask'], (0, 1), value=1)
    return tokenizer.convert_tokens_to_string(tokenizer.convert_ids_to_tokens(collect))


def check_solution(prompt, num_tokens, model_A, model_B, tokenizer, max_length=50):
    print(f"{'>'*20}\n\tPrompt\n{'<'*20}\n{prompt}\n\n")

    model_inputs = tokenizer(prompt, return_tensors='pt')

    try:
        print(f"{'>'*30}\n\tModel_A Generation\n{'<'*30}")
        print(__generate(model_A, model_inputs, num_tokens, tokenizer, max_length))
    except Exception as e:
        print(f"Error with Model_A: {e}")

    try:
        model_inputs = tokenizer(prompt, return_tensors='pt')
        print(f"\n\n{'>'*30}\n\tModel_B Generation\n{'<'*30}")
        print(__generate(model_B, model_inputs, num_tokens, tokenizer, max_length))
    except Exception as e:
        print(f"Error with Model_B: {e}")


class smolConfig:
    vocab_size = 49152
    hidden_size = 576
    intermediate_size = 1536
    num_hidden_layers = 30
    num_heads = 9
    kv_heads = 3

import torch


# Load tokenizer and reference model
checkpoint = "HuggingFaceTB/SmolLM-135M"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
reference_model = AutoModelForCausalLM.from_pretrained(checkpoint)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [118]:
# Initialize smolLM
config = smolConfig()
test_model = smolLM(config)
# Load weights
state_dict = torch.load("/content/C4AI_SMOLLM135/BareBones_SmolLM-135M.pt")
test_model.load_state_dict(state_dict, strict=False)

check_solution(prompt="Given the following film movie by a critic, rate it out of 10. Respond in a single number.\n\nThe movie started off extremely well, but just got worse after that.\nThe storyline was all over the place and everyone acted terribly.\n 10/10 would not recommend! \n\n ",
               num_tokens=1,
               model_A=reference_model,
               model_B=test_model, tokenizer=tokenizer)



>>>>>>>>>>>>>>>>>>>>
	Prompt
<<<<<<<<<<<<<<<<<<<<
Given the following film movie by a critic, rate it out of 10. Respond in a single number.

The movie started off extremely well, but just got worse after that.
The storyline was all over the place and everyone acted terribly.
 10/10 would not recommend! 

 


>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
	Model_A Generation
<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
1


>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>
	Model_B Generation
<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
coded


In [42]:
# !git lfs install
# !git clone https://huggingface.co/dsouzadaniel/C4AI_SMOLLM135
# !mv C4AI_SMOLLM135/BareBones_SmolLM-135M.pt